# Retail Demand Forecasting & Inventory Intelligence
## Phase 4 - Machine-learning demand forecasting: validation, model comparison and the final 16-day forecast

**Business question.** Can a machine-learning model forecast unit sales for 54 stores x 33 product families, 1-16 days ahead, *consistently* better than the simple rule a planner would use today - "sell what you sold on the same weekday over the last four weeks"?

**What this notebook does.** It loads the leakage-safe modeling table from Phase 3, reproduces that baseline, trains three model families (Random Forest, LightGBM, XGBoost) on five embargoed, time-ordered validation folds, analyses where and why the best model is wrong, runs controlled feature ablations, evaluates the dormant-series zero rule, selects **one** final strategy, refits it on all history before 2017-08-15 and forecasts 2017-08-16 -> 2017-08-31.

**Ground rules carried over from Phase 3** (and enforced in code):

* no random splits, no shuffling - every split is by *forecast origin*, in time order, with a 16-day embargo;
* no information dated after the forecast origin enters any feature (proved in Phase 3 by a "poison-the-future" test);
* rows with `rule_zero == 1` (dormant / never-sold series) are never used to train a model - they are forecast as 0 units - but they **are** scored;
* the metric is **RMSLE** = RMSE on `y_log1p = log(1 + sales)`; the baseline to beat is **0.469** (mean over five folds).

**Runtime note.** The heavy experiments (~1.5 h on 2 CPU cores) are cached in `phase4/cache/` (content-hashed); re-running the notebook re-uses them, so the timings shown are the ones measured when each model was trained. Delete the folder to recompute everything.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, os, time, hashlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import favorita as fv, forecasting as fc
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
fv.set_style()
BLUE, ORANGE, MUTED, INK, INK2 = fv.BLUE, fv.ORANGE, fv.MUTED, fv.INK, fv.INK2
AQUA, NEUTRAL = "#1baf7a", "#c3c2b7"
os.makedirs("figs", exist_ok=True); os.makedirs("phase4/outputs/models", exist_ok=True)
def finish(fig, name):
    fig.tight_layout(rect=[0, 0, 1, 0.94] if fig._suptitle is not None else None)
    fig.savefig(f"figs/{name}.png", dpi=110); plt.show()
def ftitle(fig, text):
    fig.suptitle(text, x=0.01, ha="left", fontsize=12.5, fontweight="bold")

## 1. Load and validate the modeling assets
Nothing is recomputed from the raw CSVs here: Phase 4 starts from the Phase 3 data products. Before any model sees the data we verify that the assets are what Phase 3 promised - and we take a *fingerprint* of the table so that the last section can prove it was never modified.

In [2]:
ASSETS = fc.find_assets()
cfg = json.load(open(f"{ASSETS}/model_config.json"))
fdict = pd.read_csv(f"{ASSETS}/feature_dictionary.csv")
lookups = pd.read_csv(f"{ASSETS}/lookups.csv")
activity = pd.read_csv(f"{ASSETS}/store_family_activity_status.csv")
tf = pd.read_parquet(f"{ASSETS}/test_features.parquet")
mt = fc.load_table(ASSETS)
FINGERPRINT = (mt.shape, round(float(mt.y_log1p.sum()), 3), round(float(mt.sd_mean_4.sum()), 3), int(mt.rule_zero.sum()))
T1, T12, T3, ALL, CAT = cfg["features_tier1"], cfg["features_tier1_2"], cfg["features_tier3_ablation"], cfg["features_all"], cfg["categorical"]

checks = []
def chk(name, actual, ok, expected=""):
    checks.append({"check": name, "actual": actual, "expected": expected, "ok": bool(ok)})
chk("modeling table rows", f"{len(mt):,}", len(mt) == 4_918_427, "4,918,427")
chk("modeling table columns", mt.shape[1], mt.shape[1] == 93, "93")
chk("features_all in table / dictionary", len(ALL), set(ALL) <= set(mt.columns) and set(ALL) == set(fdict.feature) and len(ALL) == 88, "88, all present")
chk("tiers: T1 within T1+2; T1+2 + T3 = all; no overlap", f"{len(T1)} / {len(T12)} / {len(T3)}", set(T1) <= set(T12) and set(T12) | set(T3) == set(ALL) and not set(T12) & set(T3), "29 / 78 / 10")
chk("categorical columns present", len(CAT), set(CAT) <= set(mt.columns), "9")
chk("target y_log1p available (no NaN) and = log1p(y)", int(mt.y_log1p.isna().sum()), mt.y_log1p.notna().all() and np.allclose(mt.y_log1p, np.log1p(mt.y), atol=1e-4), "0 NaN")
chk("horizons", f"{mt.horizon.min()}..{mt.horizon.max()}", sorted(mt.horizon.unique()) == list(range(1, 17)), "1..16")
chk("origins are Tuesdays; range", f"{mt.origin_date.min().date()} -> {mt.origin_date.max().date()}", (mt.origin_date.dt.dayofweek == 1).all() and mt.origin_date.nunique() == 228, "228 Tuesdays")
chk("rule_zero in {0,1}; share of rows", f"{mt.rule_zero.mean():.1%}", set(mt.rule_zero.unique()) <= {0, 1}, "binary")
chk("test features: rows", f"{len(tf):,}", len(tf) == 28_512, "28,512")
chk("test features: origin", ", ".join(tf.origin_date.dt.strftime("%Y-%m-%d").unique()), (tf.origin_date == "2017-08-15").all(), "2017-08-15 only")
chk("test features: horizons and target dates", f"{tf.horizon.min()}..{tf.horizon.max()} | {tf.target_date.min().date()} -> {tf.target_date.max().date()}",
    sorted(tf.horizon.unique()) == list(range(1, 17)) and tf.target_date.min() == pd.Timestamp("2017-08-16") and tf.target_date.max() == pd.Timestamp("2017-08-31"), "1..16 | Aug 16-31")
chk("test features: 54 stores x 33 families x 16 days, unique ids, y unknown", f"{tf.id.nunique():,} ids", tf.id.is_unique and tf.groupby("horizon").size().eq(54 * 33).all() and tf.y.isna().all(), "1,782 per day")
chk("test features contain every model feature", len(ALL), set(ALL) <= set(tf.columns), "all 88")
chk = pd.DataFrame(checks)
assert chk.ok.all(), chk[~chk.ok]
print(f"all {len(chk)} asset checks passed | fingerprint {FINGERPRINT}")
chk.drop(columns="ok")

all 14 asset checks passed | fingerprint ((4918427, 93), 18100720.0, 2164496896.0, 159051)


,check,actual,expected
0,modeling table rows,"4,918,427","4,918,427"
1,modeling table columns,93,93
2,features_all in table / dictionary,88,"88, all present"
3,tiers: T1 within T1+2; T1+2 + T3 = all; no ove...,29 / 78 / 10,29 / 78 / 10
4,categorical columns present,9,9
5,target y_log1p available (no NaN) and = log1p(y),0,0 NaN
6,horizons,1..16,1..16
7,origins are Tuesdays; range,2013-04-02 -> 2017-08-08,228 Tuesdays
8,"rule_zero in {0,1}; share of rows",3.2%,binary
9,test features: rows,"28,512","28,512"


In [3]:
print(f"rule_zero rows in the modeling table: {int(mt.rule_zero.sum()):,} ({mt.rule_zero.mean():.1%}) | at the real origin: {int(tf.rule_zero.sum()):,} rows = {int(tf.rule_zero.sum() // 16)} series")
print("categorical columns:", CAT)
print("activity status at the real origin:", activity.status.value_counts().to_dict())
print("modeling table memory: %.2f GB" % (mt.memory_usage(deep=True).sum() / 1e9))

rule_zero rows in the modeling table: 159,051 (3.2%) | at the real origin: 1,280 rows = 80 series
categorical columns: ['store_nbr', 'family_id', 'store_type', 'store_cluster', 'city_id', 'state_id', 'hol_type', 'event', 'dow']
activity status at the real origin: {'regular': 1544, 'intermittent': 158, 'never sold': 53, 'dormant': 27}
modeling table memory: 1.46 GB


## 2. Reproduce the baseline
The benchmark is the seasonal-naive forecast a planner could compute in a spreadsheet: **the mean of the last four same-weekday observations** (`sd_mean_4`; if a series has none, fall back to its 28-day mean, then to 0). It is scored exactly like a model: RMSLE over *all* validation rows of each fold.

In [4]:
BASE_PRED = {}; rows = []
for fold in fv.FOLD_ORIGINS:
    _, va_idx, _ = fc.fold_frames(mt, fold)
    d = mt.iloc[va_idx]
    BASE_PRED[fold] = fc.baseline_log(d)
    rows.append({"fold": fold, "origin": fv.FOLD_ORIGINS[fold], "validation rows": len(d), "RMSLE": fc.rmsle_from_log(d.y_log1p.to_numpy(), BASE_PRED[fold])})
bs = pd.DataFrame(rows).set_index("fold")
BASE = [{"fold": f, "rmsle": r, "train_s": 0.0, "infer_s": 0.0} for f, r in bs.RMSLE.items()]
BASE_MEAN = bs.RMSLE.mean()
print(f"mean RMSLE {BASE_MEAN:.4f} | median {bs.RMSLE.median():.4f} | std {bs.RMSLE.std(ddof=0):.4f} | best fold {bs.RMSLE.idxmin()} ({bs.RMSLE.min():.4f}) | worst fold {bs.RMSLE.idxmax()} ({bs.RMSLE.max():.4f})")
assert abs(BASE_MEAN - 0.469) < 0.001, "baseline does not reproduce the Phase 3 figure"
bs.style.format({"RMSLE": "{:.4f}", "validation rows": "{:,.0f}"}).highlight_max(subset="RMSLE", color="#fbd9cb").highlight_min(subset="RMSLE", color="#cde2fb")

mean RMSLE 0.4693 | median 0.4667 | std 0.0217 | best fold F4_recent (0.4387) | worst fold F5_latest (0.5063)


,origin,validation rows,RMSLE
fold,,,
F1_seasonal_twin,2016-08-16,"26,272",0.4667
F2_holiday_heavy,2017-04-04,"27,984",0.4634
F3_mid_2017,2017-05-30,"28,512",0.4711
F4_recent,2017-07-11,"28,512",0.4387
F5_latest,2017-07-25,"28,512",0.5063


The baseline reproduces the Phase 3 figure (0.469). This is the **minimum performance bar**: a model that does not beat it *in every fold* is not worth deploying.

## 3. Validation design - five embargoed, time-ordered folds
Each fold is one real forecast: pick a Tuesday origin, train only on rows whose 16-day targets end *before* that origin (16-day embargo), predict the following 16 days, score against what happened. Folds are chosen for different conditions: last year's back-to-school season (F1, the same season as the real test window), a Good-Friday window (F2), and three recent windows (F3-F5). The split function is the Phase 3 `favorita.split_fold` - reused unchanged.

In [5]:
rows = []
for fold, vo in fv.FOLD_ORIGINS.items():
    trm, vam = fv.split_fold(mt, vo)
    tr_idx, va_idx, va_mod = fc.fold_frames(mt, fold)
    td = mt.target_date.to_numpy()
    assert td[trm].max() < np.datetime64(vo), "embargo violated"
    assert not (trm & vam).any() and mt.origin_date.to_numpy()[va_idx].max() == np.datetime64(vo)
    assert set(mt.rule_zero.to_numpy()[tr_idx]) == {0}, "a rule_zero row leaked into training"
    rows.append({"fold": fold, "validation origin": vo, "validation window": f"{pd.Timestamp(vo) + pd.Timedelta(days=1):%Y-%m-%d} -> {pd.Timestamp(vo) + pd.Timedelta(days=16):%Y-%m-%d}",
                 "training origins": f"{mt.origin_date.to_numpy()[tr_idx].min().astype('datetime64[D]')} -> {mt.origin_date.to_numpy()[tr_idx].max().astype('datetime64[D]')}",
                 "latest training target": str(td[tr_idx].max().astype("datetime64[D]")), "train rows (modeled)": len(tr_idx), "validation rows": len(va_idx), "of which rule_zero": len(va_idx) - len(va_mod)})
print("embargo assertions passed for all five folds; no rule_zero row in any training set")
pd.DataFrame(rows).set_index("fold").style.format({"train rows (modeled)": "{:,.0f}", "validation rows": "{:,.0f}", "of which rule_zero": "{:,.0f}"})

embargo assertions passed for all five folds; no rule_zero row in any training set


,validation origin,validation window,training origins,latest training target,train rows (modeled),validation rows,of which rule_zero
fold,,,,,,,
F1_seasonal_twin,2016-08-16,2016-08-17 -> 2016-09-01,2015-07-07 -> 2016-07-26,2016-08-11,"1,436,840","26,272",842
F2_holiday_heavy,2017-04-04,2017-04-05 -> 2017-04-20,2015-07-07 -> 2017-03-14,2017-03-30,"2,287,952","27,984","1,168"
F3_mid_2017,2017-05-30,2017-05-31 -> 2017-06-15,2015-07-07 -> 2017-05-09,2017-05-25,"2,503,088","28,512","1,168"
F4_recent,2017-07-11,2017-07-12 -> 2017-07-27,2015-07-07 -> 2017-06-20,2017-07-06,"2,666,992","28,512","1,248"
F5_latest,2017-07-25,2017-07-26 -> 2017-08-10,2015-07-07 -> 2017-07-04,2017-07-20,"2,721,504","28,512","1,264"


**Why this design - and not random cross-validation - matches the business problem.**

1. *Production is a forward-looking forecast.* On Tuesday evening the planner knows nothing about the next 16 days. Every fold reproduces exactly that: features from data up to the origin, targets strictly after it.
2. *Random K-fold leaks the future.* Rows of neighbouring days are near-duplicates (a series' rolling mean today is almost identical to yesterday's). A random split puts tomorrow's row in training and today's in validation, so the model is graded on situations it has effectively already seen - the score looks excellent and means nothing. It also lets the model learn from *later* promotion policy, holidays and price levels than it would have in production.
3. *Overlapping horizons.* A row from origin `t` has targets `t+1 ... t+16`; a row from origin `t+7` overlaps 9 of those days. Without the **16-day embargo** a training target could coincide with a validation target - a direct label leak.
4. *The world drifts.* Promotion coverage grew from 0% to 47% over the history and growth slowed year by year. Only a forward split measures how well a model copes with drift.
5. *Several origins, not one.* Retail performance depends on the season (school supplies), on holidays, on the latest level shifts. Averaging five folds (and looking at their spread) prevents choosing a model that got lucky in one window.

## 4. Feature strategy
The feature sets were defined in Phase 3 from business logic and univariate screening; Phase 4 must *prove* what each adds.

* **Tier 1 (29 features)** - horizon, weekday, pay-day, store / family identity, same-weekday lags and means, rolling means, origin-relative lags, promotion, holiday flags, zero-rate and days-since-last-sale, year-ago mean.
* **Tier 1 + 2 (78 features)** - adds volatility, growth ratios, medians, longer windows, store-level and family-level context, holiday details, calendar extras.
* **Tier 3 (10 features)** - *experimental*: lagged transactions (4), oil (2), city / state ids (2), store age (1), calendar year (1). Included only if an experiment shows a measurable gain.

In [6]:
grp = fdict.set_index("feature").group
tier_tab = pd.DataFrame({"tier 1": grp[T1].value_counts(), "tier 2": grp[[f for f in T12 if f not in T1]].value_counts(), "tier 3": grp[T3].value_counts()}).fillna(0).astype(int)
tier_tab.loc["TOTAL"] = tier_tab.sum()
T3_GROUPS = {"transactions": [f for f in T3 if "_tx_" in f or "per_tx" in f], "oil": [f for f in T3 if f.startswith("oil")], "city_state": ["city_id", "state_id"],
             "store_age": ["store_age_days"], "calendar_year": ["year"]}
assert sorted(sum(T3_GROUPS.values(), [])) == sorted(T3)
print("tier-3 experiment groups:", {k: v for k, v in T3_GROUPS.items()})
tier_tab

tier-3 experiment groups: {'transactions': ['store_tx_mean_7d', 'store_tx_mean_28d', 'store_tx_growth_7_vs_28', 'store_sales_per_tx_28d'], 'oil': ['oil_price', 'oil_pct_chg_28d'], 'city_state': ['city_id', 'state_id'], 'store_age': ['store_age_days'], 'calendar_year': ['year']}


,tier 1,tier 2,tier 3
group,,,
A. Horizon & calendar,4,5,1
B. Store & product identity,2,2,2
C. Holidays & events,4,9,0
D. Promotions,3,5,0
E. Origin-relative lags,3,1,0
F. Horizon-aligned same-weekday lags,7,0,0
G. Rolling level,3,2,0
H. Volatility,0,6,0
I. Growth & year-ago,1,5,0


## 5. Modeling protocol
* **Target:** `y_log1p`. RMSE on it *is* RMSLE; predictions are clipped at 0 units (`log1p >= 0`) and converted back with `expm1` only for the final forecast.
* **Rows:** only `rule_zero == 0` rows are used to train. Validation rows with `rule_zero == 1` are predicted as exactly 0 and still counted in the score.
* **Model selection is never done on the outer fold that is being reported by early stopping:** where early stopping is used, the stopping round is chosen on an *inner* validation set carved from the training window (its last three origins, with its own 16-day embargo).
* **Hyper-parameters** are chosen from a deliberately small grid on three of the five folds (F1, F3, F5 - the seasonal twin, a mid-2017 and the latest window) and then every model is evaluated on all five folds. This is standard practice, but note that some optimism remains because the same folds are used for tuning and reporting; the grids are tiny to keep it small.
* **Fair comparison:** all models see the same rows and the same feature sets. Random Forest needs `NaN -> -1` and is trained on a 250,000-row random subsample *of the training window* (a full forest on ~2.7M rows x 78 features is not practical on 2 CPU cores); the time-ordered split itself is unchanged.

## 6. Model A - Random Forest (tree benchmark)
A bagged-tree benchmark with a small grid over the four parameters that matter: depth, leaf size, features per split (trees = 100 in all cases).

In [7]:
TUNE_FOLDS = ["F1_seasonal_twin", "F3_mid_2017", "F5_latest"]
RF_GRID = {"rf_A": dict(n_estimators=100, max_depth=14, min_samples_leaf=50, max_features=0.5),
           "rf_B": dict(n_estimators=100, max_depth=18, min_samples_leaf=20, max_features=0.5),
           "rf_C": dict(n_estimators=100, max_depth=22, min_samples_leaf=10, max_features=0.33)}
rf_tune = {}
for name, p in RF_GRID.items():
    res = fc.five_fold("rf", mt, T1, p, folds=TUNE_FOLDS)
    rf_tune[name] = res
tab = pd.DataFrame([{"config": k, **{kk: vv for kk, vv in RF_GRID[k].items() if kk != "n_estimators"}, **{r["fold"].split("_")[0]: r["rmsle"] for r in v},
                     "mean (F1,F3,F5)": np.mean([r["rmsle"] for r in v]), "train s / fold": np.mean([r["train_s"] for r in v])} for k, v in rf_tune.items()]).set_index("config")
RF_BEST = tab["mean (F1,F3,F5)"].idxmin(); RF_PARAMS = RF_GRID[RF_BEST]
print("selected Random Forest configuration:", RF_BEST, RF_PARAMS)
tab

selected Random Forest configuration: rf_C {'n_estimators': 100, 'max_depth': 22, 'min_samples_leaf': 10, 'max_features': 0.33}


,max_depth,min_samples_leaf,max_features,F1,F3,F5,"mean (F1,F3,F5)",train s / fold
config,,,,,,,,
rf_A,14,50,0.5000,0.4060,0.4043,0.4479,0.4194,58.2587
rf_B,18,20,0.5000,0.3993,0.3983,0.4371,0.4116,68.4961
rf_C,22,10,0.3300,0.3972,0.3942,0.4346,0.4087,50.1308


In [8]:
RF_T1 = fc.five_fold("rf", mt, T1, RF_PARAMS)
RF_T12 = fc.five_fold("rf", mt, T12, RF_PARAMS)
for nm, res in [("Random Forest, tier 1", RF_T1), ("Random Forest, tier 1+2", RF_T12)]:
    print(f"{nm}: " + " | ".join(f"{r['fold'].split('_')[0]} {r['rmsle']:.4f}" for r in res) + f" | mean {np.mean([r['rmsle'] for r in res]):.4f}"
          f" | train {np.mean([r['train_s'] for r in res]):.0f}s/fold | inference {np.mean([r['infer_s'] for r in res]):.2f}s/fold")

Random Forest, tier 1: F1 0.3972 | F2 0.4186 | F3 0.3942 | F4 0.3933 | F5 0.4346 | mean 0.4076 | train 50s/fold | inference 0.32s/fold
Random Forest, tier 1+2: F1 0.3954 | F2 0.4202 | F3 0.3924 | F4 0.3912 | F5 0.4297 | mean 0.4058 | train 158s/fold | inference 0.34s/fold


## 7. Model B - LightGBM (main model)
Gradient-boosted trees are the natural fit for this table: they handle the native `NaN`s of Phase 3 (missing lags mean *"no history"*, which carries information), non-linear interactions (promotion x weekday x horizon), and millions of rows quickly. The store and family identifiers are passed to LightGBM as **categorical** features (it groups categories optimally instead of assuming an order between store numbers).

**Tuning protocol.** A small grid (3 configurations + a "categoricals as plain numbers" control) is run on the three tuning folds with **early stopping on an inner validation set** (the last three training origins, with their own 16-day embargo - the outer validation fold is never used to pick the number of rounds). The number of rounds for the five-fold runs is then fixed at 1.05 x the median stopped round.

In [9]:
LGB_BASE = dict(objective="regression", num_threads=2, max_bin=127, bagging_fraction=0.3, bagging_freq=1, lambda_l2=1.0, feature_fraction=0.8)
LGB_GRID = {"lgb_A": dict(LGB_BASE, learning_rate=0.08, num_leaves=63, min_data_in_leaf=200),
            "lgb_B": dict(LGB_BASE, learning_rate=0.08, num_leaves=127, min_data_in_leaf=100),
            "lgb_C": dict(LGB_BASE, learning_rate=0.08, num_leaves=255, min_data_in_leaf=100, feature_fraction=0.6)}
MAX_ROUNDS = 800
lgb_tune = {}
for name, p in LGB_GRID.items():
    lgb_tune[name] = fc.five_fold("lgb", mt, T1, p, rounds=MAX_ROUNDS, folds=TUNE_FOLDS, categorical=CAT, early_stop=True)
lgb_tune["lgb_B_no_categorical"] = fc.five_fold("lgb", mt, T1, LGB_GRID["lgb_B"], rounds=MAX_ROUNDS, folds=TUNE_FOLDS, categorical=(), early_stop=True)
tab = pd.DataFrame([{"config": k, "leaves": (LGB_GRID.get(k) or LGB_GRID["lgb_B"])["num_leaves"], "min_leaf": (LGB_GRID.get(k) or LGB_GRID["lgb_B"])["min_data_in_leaf"],
                     "feat_frac": (LGB_GRID.get(k) or LGB_GRID["lgb_B"])["feature_fraction"], "categorical": "no" if "no_cat" in k else "yes",
                     **{r["fold"].split("_")[0]: r["rmsle"] for r in v}, "mean (F1,F3,F5)": np.mean([r["rmsle"] for r in v]),
                     "stopped round (median)": int(np.median([r["best_iter"] for r in v])), "train s / fold": np.mean([r["train_s"] for r in v])} for k, v in lgb_tune.items()]).set_index("config")
LGB_BEST = tab.loc[list(LGB_GRID), "mean (F1,F3,F5)"].idxmin(); LGB_PARAMS = LGB_GRID[LGB_BEST]
LGB_CAT = CAT if tab.loc[LGB_BEST, "mean (F1,F3,F5)"] <= tab.loc["lgb_B_no_categorical", "mean (F1,F3,F5)"] + 0.0005 or LGB_BEST != "lgb_B" else ()
print("selected LightGBM configuration:", LGB_BEST, "| categorical handling:", "native categorical" if LGB_CAT else "numeric codes")
tab.style.format({c: "{:.4f}" for c in ["F1", "F3", "F5", "mean (F1,F3,F5)"]} | {"train s / fold": "{:.0f}"})

selected LightGBM configuration: lgb_C | categorical handling: native categorical


,leaves,min_leaf,feat_frac,categorical,F1,F3,F5,"mean (F1,F3,F5)",stopped round (median),train s / fold
config,,,,,,,,,,
lgb_A,63,200,0.800000,yes,0.3910,0.3873,0.4113,0.3965,227,40
lgb_B,127,100,0.800000,yes,0.3910,0.3880,0.4104,0.3965,188,41
lgb_C,255,100,0.600000,yes,0.3905,0.3860,0.4103,0.3956,127,35
lgb_B_no_categorical,127,100,0.800000,no,0.3917,0.3885,0.4093,0.3965,213,40


Early stopping picks the number of rounds for each feature set (tier 1 and tier 1+2 are stopped separately, tier 2 with half the features tried per tree because there are more of them).

In [10]:
LGB_P12 = dict(LGB_PARAMS, feature_fraction=0.5)
es12 = fc.five_fold("lgb", mt, T12, LGB_P12, rounds=MAX_ROUNDS, folds=TUNE_FOLDS, categorical=LGB_CAT, early_stop=True)
es1 = lgb_tune[LGB_BEST] if LGB_CAT == CAT else fc.five_fold("lgb", mt, T1, LGB_PARAMS, rounds=MAX_ROUNDS, folds=TUNE_FOLDS, categorical=LGB_CAT, early_stop=True)
ROUNDS_T1 = int(1.05 * np.median([r["best_iter"] for r in es1])); ROUNDS_T12 = int(1.05 * np.median([r["best_iter"] for r in es12]))
print(f"stopped rounds tier 1: {[r['best_iter'] for r in es1]} -> fixed {ROUNDS_T1} | tier 1+2: {[r['best_iter'] for r in es12]} -> fixed {ROUNDS_T12}")
LGB_T1 = fc.five_fold("lgb", mt, T1, LGB_PARAMS, rounds=ROUNDS_T1, categorical=LGB_CAT)
LGB_T12 = fc.five_fold("lgb", mt, T12, LGB_P12, rounds=ROUNDS_T12, categorical=LGB_CAT)
def line(nm, res):
    print(f"{nm}: " + " | ".join(f"{r['fold'].split('_')[0]} {r['rmsle']:.4f}" for r in res) + f" | mean {np.mean([r['rmsle'] for r in res]):.4f}"
          f" | train {np.mean([r['train_s'] for r in res]):.0f}s/fold | inference {np.mean([r['infer_s'] for r in res]):.1f}s/fold")
line("LightGBM tier 1  ", LGB_T1); line("LightGBM tier 1+2", LGB_T12)

stopped rounds tier 1: [84, 240, 127] -> fixed 133 | tier 1+2: [74, 165, 208] -> fixed 173
LightGBM tier 1  : F1 0.3890 | F2 0.4171 | F3 0.3843 | F4 0.3938 | F5 0.4059 | mean 0.3980 | train 27s/fold | inference 0.2s/fold
LightGBM tier 1+2: F1 0.3874 | F2 0.4159 | F3 0.3834 | F4 0.3882 | F5 0.4055 | mean 0.3961 | train 45s/fold | inference 0.3s/fold


## 8. Model C - XGBoost (optional cross-check)
A second boosting implementation on the same rows, folds and features. NaNs are handled natively by XGBoost as well. Two small configurations are compared on the tuning folds with a fixed 300 rounds (no early stopping - XGBoost is only a cross-check here).

In [11]:
XGB_BASE = dict(objective="reg:squarederror", tree_method="hist", nthread=2, max_bin=127, reg_lambda=1.0, subsample=0.5, colsample_bytree=0.8)
XGB_GRID = {"xgb_A": dict(XGB_BASE, max_depth=6, eta=0.10, min_child_weight=100),
            "xgb_B": dict(XGB_BASE, max_depth=8, eta=0.08, min_child_weight=50)}
XGB_ROUNDS = 300
xgb_tune = {k: fc.five_fold("xgb", mt, T1, p, rounds=XGB_ROUNDS, folds=TUNE_FOLDS) for k, p in XGB_GRID.items()}
tab = pd.DataFrame([{"config": k, "depth": XGB_GRID[k]["max_depth"], "eta": XGB_GRID[k]["eta"], **{r["fold"].split("_")[0]: r["rmsle"] for r in v},
                     "mean (F1,F3,F5)": np.mean([r["rmsle"] for r in v]), "train s / fold": np.mean([r["train_s"] for r in v])} for k, v in xgb_tune.items()]).set_index("config")
XGB_BEST = tab["mean (F1,F3,F5)"].idxmin(); XGB_PARAMS = XGB_GRID[XGB_BEST]
print("selected XGBoost configuration:", XGB_BEST)
XGB_T1 = fc.five_fold("xgb", mt, T1, XGB_PARAMS, rounds=XGB_ROUNDS)
XGB_T12 = fc.five_fold("xgb", mt, T12, dict(XGB_PARAMS, colsample_bytree=0.5), rounds=XGB_ROUNDS)
line("XGBoost tier 1  ", XGB_T1); line("XGBoost tier 1+2", XGB_T12)
tab.style.format({c: "{:.4f}" for c in ["F1", "F3", "F5", "mean (F1,F3,F5)"]} | {"train s / fold": "{:.0f}"})

selected XGBoost configuration: xgb_B
XGBoost tier 1  : F1 0.3910 | F2 0.4155 | F3 0.3859 | F4 0.3950 | F5 0.4024 | mean 0.3979 | train 61s/fold | inference 0.1s/fold
XGBoost tier 1+2: F1 0.3872 | F2 0.4162 | F3 0.3830 | F4 0.3889 | F5 0.4042 | mean 0.3959 | train 112s/fold | inference 0.1s/fold


,depth,eta,F1,F3,F5,"mean (F1,F3,F5)",train s / fold
config,,,,,,,
xgb_A,6,0.100000,0.3910,0.3875,0.4045,0.3943,46
xgb_B,8,0.080000,0.3910,0.3859,0.4024,0.3931,58


## 9. Feature ablations (LightGBM, five folds)
Rule of the experiment: **one change at a time against the tier 1+2 reference**, all five folds, same rounds. A change is *kept* only if it lowers the mean RMSLE by at least 0.0005 **and** improves at least 4 of the 5 folds - otherwise the simpler set wins. This protects against adopting noise that helps one window.
Steps: (a) tier 1 vs tier 1+2, (b) the experimental tier-3 groups one by one and all together, (c) removing the features the model hardly uses (mean gain share < 0.1% in the tier 1+2 model).

In [12]:
def gain_share(results):
    g = pd.DataFrame([r["gain"] for r in results]).T
    return (g / g.sum()).mean(axis=1).sort_values(ascending=False)
GS12 = gain_share(LGB_T12)
T12_PRUNED = [f for f in T12 if GS12[f] >= 0.001]
ABL = {"tier 1 (29)": LGB_T1, "tier 1+2 (78) - reference": LGB_T12}
ABL_FEATS = {"tier 1 (29)": T1, "tier 1+2 (78) - reference": T12}
ABL_PARAMS = {"tier 1 (29)": (LGB_PARAMS, ROUNDS_T1)}
for gname, gf in [("+ transactions (lagged)", T3_GROUPS["transactions"]), ("+ oil price", T3_GROUPS["oil"]), ("+ store static (city, state, age)", T3_GROUPS["city_state"] + T3_GROUPS["store_age"]),
                  ("+ calendar year", T3_GROUPS["calendar_year"]), ("+ all tier 3 (10)", T3)]:
    feats = T12 + gf
    ABL_FEATS[gname] = feats; ABL[gname] = fc.five_fold("lgb", mt, feats, LGB_P12, rounds=ROUNDS_T12, categorical=LGB_CAT)
nm = f"tier 1+2 pruned ({len(T12_PRUNED)})"
ABL_FEATS[nm] = T12_PRUNED; ABL[nm] = fc.five_fold("lgb", mt, T12_PRUNED, LGB_P12, rounds=ROUNDS_T12, categorical=LGB_CAT)
ref = np.array([r["rmsle"] for r in LGB_T12])
rows = []
for k, res in ABL.items():
    s = np.array([r["rmsle"] for r in res]); d = ref - s          # positive = better than the reference
    rows.append({"variant": k, "features": len(ABL_FEATS[k]), **{f"F{i+1}": s[i] for i in range(5)}, "mean": s.mean(), "mean gain vs reference": d.mean(), "folds better": int((d > 0).sum())})
abl = pd.DataFrame(rows).set_index("variant")
def keep(k):
    if k.startswith("tier 1 ("): return abl.loc[k, "mean gain vs reference"] > -0.0005          # tier 1 is "as good" -> simpler wins
    if "pruned" in k: return abl.loc[k, "mean gain vs reference"] > -0.0003                       # removing weak features must not hurt
    if "reference" in k: return False
    return abl.loc[k, "mean gain vs reference"] >= 0.0005 and abl.loc[k, "folds better"] >= 4
def decision(k):
    if "reference" in k: return "-"
    if k.startswith("tier 1 ("): return "simpler set is as good" if keep(k) else "tier 2 needed"
    if "pruned" in k: return "drop weak features" if keep(k) else "keep them"
    return "KEEP (helps)" if keep(k) else "reject"
abl["decision"] = [decision(k) for k in abl.index]
abl.style.format({**{f"F{i+1}": "{:.4f}" for i in range(5)}, "mean": "{:.4f}", "mean gain vs reference": "{:+.4f}"})

,features,F1,F2,F3,F4,F5,mean,mean gain vs reference,folds better,decision
variant,,,,,,,,,,
tier 1 (29),29,0.3890,0.4171,0.3843,0.3938,0.4059,0.3980,-0.0020,0,tier 2 needed
tier 1+2 (78) - reference,78,0.3874,0.4159,0.3834,0.3882,0.4055,0.3961,+0.0000,0,-
+ transactions (lagged),82,0.3869,0.4141,0.3821,0.3884,0.4067,0.3956,+0.0004,3,reject
+ oil price,80,0.3898,0.4142,0.3826,0.3885,0.4062,0.3963,-0.0002,2,reject
"+ store static (city, state, age)",81,0.3894,0.4173,0.3830,0.3889,0.4032,0.3963,-0.0003,2,reject
+ calendar year,79,0.3878,0.4146,0.3824,0.3872,0.4062,0.3956,+0.0004,3,reject
+ all tier 3 (10),88,0.3884,0.4120,0.3821,0.3885,0.4095,0.3961,-0.0000,2,reject
tier 1+2 pruned (15),15,0.4101,0.4434,0.4026,0.4012,0.4596,0.4234,-0.0273,0,keep them


In [13]:
# choose the final feature set by the rule above: an adopted tier-3 group beats everything; otherwise the pruned set, then tier 1, then tier 1+2.
adopted = [k for k in abl.index if k.startswith("+") and keep(k)]
if adopted:
    FEAT_CHOICE = abl.loc[adopted, "mean"].idxmin()
elif keep(f"tier 1+2 pruned ({len(T12_PRUNED)})") and abl.loc[f"tier 1+2 pruned ({len(T12_PRUNED)})", "mean"] <= abl.loc["tier 1 (29)", "mean"] + 0.0005:
    FEAT_CHOICE = f"tier 1+2 pruned ({len(T12_PRUNED)})"
elif keep("tier 1 (29)"):
    FEAT_CHOICE = "tier 1 (29)"
else:
    FEAT_CHOICE = "tier 1+2 (78) - reference"
FEAT_FINAL = ABL_FEATS[FEAT_CHOICE]
FEAT_ROUNDS, FEAT_PARAMS = ((ROUNDS_T1, LGB_PARAMS) if FEAT_CHOICE.startswith("tier 1 (") else (ROUNDS_T12, LGB_P12))
LGB_FINAL = ABL[FEAT_CHOICE]
print(f"feature set chosen for the final LightGBM: {FEAT_CHOICE}  ({len(FEAT_FINAL)} features), rounds {FEAT_ROUNDS}")
fig, ax = plt.subplots(figsize=(8.5, 4.2))
d = abl["mean gain vs reference"].drop("tier 1+2 (78) - reference")
ax.barh(d.index[::-1], d.values[::-1], color=[BLUE if v > 0 else NEUTRAL for v in d.values[::-1]], height=0.6)
ax.axvline(0, color=INK2, lw=0.8); ax.axvline(0.0005, color=ORANGE, lw=1, ls="--"); ax.text(0.0005, len(d) - 0.35, " keep threshold", color=ORANGE, fontsize=8.5, va="bottom")
ax.set_xlabel("mean RMSLE improvement vs tier 1+2 (positive = better)"); ftitle(fig, "Feature ablations - change in mean RMSLE against the tier 1+2 reference")
finish(fig, "p4_ablation")

feature set chosen for the final LightGBM: tier 1+2 (78) - reference  (78 features), rounds 173


**Residual-target variant.** Instead of forecasting `log1p(sales)` from scratch the model can forecast the *correction to the baseline*: `y_log1p - log1p(baseline)`. It is a common way to inject the seasonal-naive prior into a tree model; we test it once on the chosen feature set.

In [14]:
LGB_RESID = fc.five_fold("lgb", mt, FEAT_FINAL, FEAT_PARAMS, rounds=FEAT_ROUNDS, categorical=LGB_CAT, target="residual")
line("LightGBM, residual target", LGB_RESID); line("LightGBM, direct target  ", LGB_FINAL)

LightGBM, residual target: F1 0.3912 | F2 0.4123 | F3 0.3846 | F4 0.3892 | F5 0.4026 | mean 0.3960 | train 45s/fold | inference 0.3s/fold
LightGBM, direct target  : F1 0.3874 | F2 0.4159 | F3 0.3834 | F4 0.3882 | F5 0.4055 | mean 0.3961 | train 45s/fold | inference 0.3s/fold


## 10. Model comparison and selection
Every candidate is scored on the same five folds. Besides the mean we look at **the worst fold** (does it ever lose to the baseline?), **the two most recent folds** (F4, F5 - closest to production), **F1 - the seasonal twin** of the real forecast window (mid-August 2016), the fold-to-fold **spread**, and practicality (training and inference time).

In [15]:
def blend(a, b):
    out = []
    for ra, rb in zip(a, b):
        assert ra["fold"] == rb["fold"] and (ra["va_idx"] == rb["va_idx"]).all()
        p = 0.5 * ra["pred"].astype(np.float64) + 0.5 * rb["pred"].astype(np.float64)
        out.append({**{k: ra[k] for k in ["fold", "va_idx"]}, "pred": p.astype(np.float32), "rmsle": fc.rmsle_from_log(mt.y_log1p.to_numpy()[ra["va_idx"]], p),
                    "train_s": ra["train_s"] + rb["train_s"], "infer_s": ra["infer_s"] + rb["infer_s"]})
    return out
XGB_FINAL = XGB_T12 if np.mean([r["rmsle"] for r in XGB_T12]) < np.mean([r["rmsle"] for r in XGB_T1]) else XGB_T1
XGB_FINAL_FEATS = T12 if XGB_FINAL is XGB_T12 else T1
BLEND = blend(LGB_FINAL, XGB_FINAL)
RF_FINAL = RF_T12 if np.mean([r["rmsle"] for r in RF_T12]) < np.mean([r["rmsle"] for r in RF_T1]) else RF_T1
CANDS = {"Baseline (mean of last 4 same weekdays)": BASE, "Random Forest, tier 1": RF_T1, "Random Forest, tier 1+2": RF_T12,
         "LightGBM, tier 1": LGB_T1, "LightGBM, tier 1+2": LGB_T12, f"LightGBM, {FEAT_CHOICE} (ablation-selected)": LGB_FINAL, "LightGBM, residual target": LGB_RESID,
         "XGBoost, tier 1": XGB_T1, "XGBoost, tier 1+2": XGB_T12, "Blend 50/50: LightGBM (selected) + XGBoost": BLEND}
# de-duplicate identical candidates (the ablation-selected set can equal tier 1 or tier 1+2)
seen = {};
for k, v in list(CANDS.items()):
    sig = tuple(round(r["rmsle"], 9) for r in v)
    if sig in seen and k != "Baseline (mean of last 4 same weekdays)": CANDS.pop(k)
    else: seen[sig] = k
rows = []
for k, res in CANDS.items():
    s = np.array([r["rmsle"] for r in res])
    rows.append({"Model": k, **{f"F{i+1}": s[i] for i in range(5)}, "Mean RMSLE": s.mean(), "Improvement vs Baseline": 1 - s.mean() / BASE_MEAN, "Worst fold": s.max(), "F4+F5 mean": s[3:].mean(), "Std": s.std(),
                 "Folds beating baseline": int((s < np.array([r["rmsle"] for r in BASE])).sum()), "Train s/fold": np.mean([r["train_s"] for r in res]), "Infer s/fold": np.mean([r["infer_s"] for r in res])})
CMP = pd.DataFrame(rows).set_index("Model")
CMP.style.format({**{f"F{i+1}": "{:.4f}" for i in range(5)}, "Mean RMSLE": "{:.4f}", "Improvement vs Baseline": "{:.1%}", "Worst fold": "{:.4f}", "F4+F5 mean": "{:.4f}", "Std": "{:.4f}", "Train s/fold": "{:.0f}", "Infer s/fold": "{:.1f}"}).highlight_min(subset=["Mean RMSLE"], color="#cde2fb")

,F1,F2,F3,F4,F5,Mean RMSLE,Improvement vs Baseline,Worst fold,F4+F5 mean,Std,Folds beating baseline,Train s/fold,Infer s/fold
Model,,,,,,,,,,,,,
Baseline (mean of last 4 same weekdays),0.4667,0.4634,0.4711,0.4387,0.5063,0.4693,0.0%,0.5063,0.4725,0.0217,0,0,0.0
"Random Forest, tier 1",0.3972,0.4186,0.3942,0.3933,0.4346,0.4076,13.1%,0.4346,0.4140,0.0164,5,50,0.3
"Random Forest, tier 1+2",0.3954,0.4202,0.3924,0.3912,0.4297,0.4058,13.5%,0.4297,0.4104,0.0160,5,158,0.3
"LightGBM, tier 1",0.3890,0.4171,0.3843,0.3938,0.4059,0.3980,15.2%,0.4171,0.3999,0.0119,5,27,0.2
"LightGBM, tier 1+2",0.3874,0.4159,0.3834,0.3882,0.4055,0.3961,15.6%,0.4159,0.3968,0.0125,5,45,0.3
"LightGBM, residual target",0.3912,0.4123,0.3846,0.3892,0.4026,0.3960,15.6%,0.4123,0.3959,0.0101,5,45,0.3
"XGBoost, tier 1",0.3910,0.4155,0.3859,0.3950,0.4024,0.3979,15.2%,0.4155,0.3987,0.0103,5,61,0.1
"XGBoost, tier 1+2",0.3872,0.4162,0.3830,0.3889,0.4042,0.3959,15.6%,0.4162,0.3966,0.0124,5,112,0.1
Blend 50/50: LightGBM (selected) + XGBoost,0.3852,0.4139,0.3812,0.3874,0.4034,0.3942,16.0%,0.4139,0.3954,0.0124,5,157,0.4


In [16]:
eligible = CMP[(CMP["Folds beating baseline"] == 5) & (CMP.index != "Baseline (mean of last 4 same weekdays)")]
ranked = eligible.sort_values("Mean RMSLE")
print(ranked[["Mean RMSLE", "Worst fold", "F4+F5 mean", "F1", "Std", "Train s/fold"]].round(4).to_string())
WINNER = ranked.index[0]
# practicality tie-break: if a simpler single model is within 0.0007 of the leader, prefer it (blend > one model in cost)
if WINNER.startswith("Blend") and ranked.iloc[0]["Mean RMSLE"] > ranked[~ranked.index.str.startswith("Blend")].iloc[0]["Mean RMSLE"] - 0.0007:
    WINNER = ranked[~ranked.index.str.startswith("Blend")].index[0]
print("\nSELECTED:", WINNER)
show = [("Baseline", BASE, NEUTRAL), ("Random Forest", RF_FINAL, ORANGE), ("XGBoost", XGB_FINAL, AQUA), ("LightGBM", LGB_FINAL, BLUE)]
fig, ax = plt.subplots(figsize=(10, 4.4)); w = 0.2; x = np.arange(6)
for j, (nm_, res, col) in enumerate(show):
    v = [r["rmsle"] for r in res] + [np.mean([r["rmsle"] for r in res])]
    ax.bar(x + (j - 1.5) * w, v, w * 0.9, color=col, label=nm_)
    for xi, vi in zip(x, v): ax.text(xi + (j - 1.5) * w, vi + 0.004, f"{vi:.3f}", ha="center", fontsize=6.8, color=INK2, rotation=90)
ax.set_xticks(x); ax.set_xticklabels(["F1 seasonal\ntwin '16", "F2 holiday\nApr '17", "F3 May '17", "F4 Jul '17", "F5 latest", "MEAN"]); ax.set_ylabel("RMSLE (lower is better)"); ax.set_ylim(0, ax.get_ylim()[1] * 1.08)
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.0)); ftitle(fig, "RMSLE by validation fold - baseline vs the three model families")
finish(fig, "p4_model_comparison")

                                            Mean RMSLE  Worst fold  F4+F5 mean     F1    Std  Train s/fold
Model                                                                                                     
Blend 50/50: LightGBM (selected) + XGBoost      0.3942      0.4139      0.3954 0.3852 0.0124      156.7418
XGBoost, tier 1+2                               0.3959      0.4162      0.3966 0.3872 0.0124      111.7230
LightGBM, residual target                       0.3960      0.4123      0.3959 0.3912 0.0101       45.2399
LightGBM, tier 1+2                              0.3961      0.4159      0.3968 0.3874 0.0125       45.0188
XGBoost, tier 1                                 0.3979      0.4155      0.3987 0.3910 0.0103       61.4412
LightGBM, tier 1                                0.3980      0.4171      0.3999 0.3890 0.0119       27.4038
Random Forest, tier 1+2                         0.4058      0.4297      0.4104 0.3954 0.0160      157.7889
Random Forest, tier 1                

## 11. Error analysis of the selected model
The five validation folds are pooled (each row is one real out-of-time forecast: one store x family x day). For every segment we report **RMSLE** of the model and of the baseline, the segment's **share of the total squared error** (where the error mass is), and two unit-level diagnostics: **WAPE** = sum|forecast - actual| / sum(actual) and **bias** = sum(forecast - actual) / sum(actual) (negative = under-forecasting). RMSLE is the model-selection metric; WAPE / bias tell an inventory planner whether the units are right.

In [17]:
SPECS = {}
def spec(kind, feats, params, rounds=None, **kw): return dict(kind=kind, features=list(feats), params=params, rounds=rounds, kw=kw)
SPECS["Random Forest, tier 1"] = [spec("rf", T1, RF_PARAMS)]; SPECS["Random Forest, tier 1+2"] = [spec("rf", T12, RF_PARAMS)]
SPECS["LightGBM, tier 1"] = [spec("lgb", T1, LGB_PARAMS, ROUNDS_T1, categorical=list(LGB_CAT))]
SPECS["LightGBM, tier 1+2"] = [spec("lgb", T12, LGB_P12, ROUNDS_T12, categorical=list(LGB_CAT))]
LGB_SPEC = spec("lgb", FEAT_FINAL, FEAT_PARAMS, FEAT_ROUNDS, categorical=list(LGB_CAT))
SPECS[f"LightGBM, {FEAT_CHOICE} (ablation-selected)"] = [LGB_SPEC]
SPECS["LightGBM, residual target"] = [dict(LGB_SPEC, kw=dict(LGB_SPEC["kw"], target="residual"))]
SPECS["XGBoost, tier 1"] = [spec("xgb", T1, XGB_PARAMS, XGB_ROUNDS)]; SPECS["XGBoost, tier 1+2"] = [spec("xgb", T12, dict(XGB_PARAMS, colsample_bytree=0.5), XGB_ROUNDS)]
SPECS["Blend 50/50: LightGBM (selected) + XGBoost"] = [LGB_SPEC, spec("xgb", XGB_FINAL_FEATS, dict(XGB_PARAMS, colsample_bytree=0.5) if XGB_FINAL_FEATS is T12 else XGB_PARAMS, XGB_ROUNDS)]
WIN_SPEC = SPECS[WINNER]; WIN_RES = CANDS[WINNER]
LOOK = pd.read_csv(f"{ASSETS}/lookups.csv"); FAM = LOOK[LOOK.column == "family_id"].set_index("code").label
Y_U = np.expm1(mt.y_log1p.to_numpy())

def build_val(res):
    parts = []
    for r in res:
        i = r["va_idx"]; d = mt.iloc[i]
        parts.append(pd.DataFrame({"fold": r["fold"].split("_")[0], "origin_date": d.origin_date.to_numpy(), "target_date": d.target_date.to_numpy(), "horizon": d.horizon.to_numpy(), "store_nbr": d.store_nbr.to_numpy(),
                                   "family": d.family_id.map(FAM).to_numpy(), "dow": d.dow.to_numpy(), "onpromotion": d.onpromotion.to_numpy(), "promo_flag": d.promo_flag.to_numpy(), "hol_any": d.hol_any.to_numpy(),
                                   "rule_zero": d.rule_zero.to_numpy(), "zero_rate_28d": d.roll_zero_rate_28d.to_numpy(), "level_28d": d.roll_mean_28d.to_numpy(),
                                   "y_log": d.y_log1p.to_numpy(), "pred_log": r["pred"].astype(np.float64), "base_log": BASE_PRED[r["fold"]]}))
    v = pd.concat(parts, ignore_index=True)
    v["se"] = (v.pred_log - v.y_log) ** 2; v["se_base"] = (v.base_log - v.y_log) ** 2
    v["y_u"] = np.expm1(v.y_log); v["pred_u"] = np.expm1(v.pred_log); v["base_u"] = np.expm1(v.base_log)
    return v
VAL = build_val(WIN_RES)
assert abs(np.sqrt(VAL.groupby("fold").se.mean()).mean() - CMP.loc[WINNER, "Mean RMSLE"]) < 1e-6
VAL["activity"] = np.where(VAL.rule_zero == 1, "inactive (rule_zero)", np.where(VAL.zero_rate_28d.isna(), "no history", np.where(VAL.zero_rate_28d >= 0.5, "intermittent", "regular")))
VAL["volume"] = pd.cut(VAL.level_28d, [-1, 1, 5, 20, 100, np.inf], labels=["<=1 unit/day", "1-5", "5-20", "20-100", ">100"]).astype(object).fillna("no history")
VAL["promo"] = np.where(VAL.onpromotion > 0, "on promotion", "no promotion")
VAL["weekday"] = VAL.dow.map({0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"})
VAL["holiday"] = np.where(VAL.hol_any > 0, "holiday / event day", "regular day")
TOTAL_SE = VAL.se.sum()
def seg(col, order=None):
    g = VAL.groupby(col)
    t = pd.DataFrame({"rows": g.size(), "row share": g.size() / len(VAL), "RMSLE model": np.sqrt(g.se.mean()), "RMSLE baseline": np.sqrt(g.se_base.mean()), "share of model SSE": g.se.sum() / TOTAL_SE,
                      "WAPE": g.apply(lambda x: (x.pred_u - x.y_u).abs().sum() / x.y_u.sum()), "bias": g.apply(lambda x: (x.pred_u.sum() - x.y_u.sum()) / x.y_u.sum())})
    t["gain vs baseline"] = 1 - t["RMSLE model"] / t["RMSLE baseline"]
    if order:
        order = [o for o in order if o in t.index]
    return t.loc[order] if order else t
FMT = {"rows": "{:,.0f}", "row share": "{:.1%}", "RMSLE model": "{:.4f}", "RMSLE baseline": "{:.4f}", "share of model SSE": "{:.1%}", "WAPE": "{:.1%}", "bias": "{:+.1%}", "gain vs baseline": "{:+.1%}"}
overall = pd.DataFrame({"RMSLE model": [np.sqrt(VAL.se.mean())], "RMSLE baseline": [np.sqrt(VAL.se_base.mean())], "WAPE": [(VAL.pred_u - VAL.y_u).abs().sum() / VAL.y_u.sum()], "bias": [(VAL.pred_u.sum() - VAL.y_u.sum()) / VAL.y_u.sum()]}, index=["pooled 5 folds"])
print("Selected model:", WINNER); overall.style.format({"RMSLE model": "{:.4f}", "RMSLE baseline": "{:.4f}", "WAPE": "{:.1%}", "bias": "{:+.1%}"})

Selected model: Blend 50/50: LightGBM (selected) + XGBoost


,RMSLE model,RMSLE baseline,WAPE,bias
pooled 5 folds,0.3945,0.4698,12.6%,-1.5%


### 11.1 By forecast horizon and by validation fold

In [18]:
H = seg("horizon"); FD = seg("fold")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.9), gridspec_kw={"width_ratios": [1.6, 1]})
axs[0].plot(H.index, H["RMSLE baseline"], color=NEUTRAL, lw=2, marker="o", ms=4, label="baseline"); axs[0].plot(H.index, H["RMSLE model"], color=BLUE, lw=2, marker="o", ms=4, label="selected model")
axs[0].set_xlabel("forecast horizon (days ahead)"); axs[0].set_ylabel("RMSLE"); axs[0].set_xticks(range(1, 17)); axs[0].legend(frameon=False); axs[0].set_title("Error grows with the horizon, the gap to the baseline stays", loc="left", fontsize=10)
axs[1].bar(FD.index, FD["RMSLE baseline"], color=NEUTRAL, width=0.4, align="edge", label="baseline"); axs[1].bar(FD.index, FD["RMSLE model"], color=BLUE, width=-0.4, align="edge", label="model")
axs[1].set_ylabel("RMSLE"); axs[1].set_title("By fold", loc="left", fontsize=10); axs[1].legend(frameon=False)
ftitle(fig, "Error by horizon and by validation fold"); finish(fig, "p4_error_horizon")
H.style.format(FMT)

,rows,row share,RMSLE model,RMSLE baseline,share of model SSE,WAPE,bias,gain vs baseline
horizon,,,,,,,,
1,"8,759",6.3%,0.3903,0.4528,6.1%,10.2%,-1.9%,+13.8%
2,"8,759",6.3%,0.3944,0.4660,6.3%,11.1%,-2.9%,+15.4%
3,"8,759",6.3%,0.3895,0.4409,6.1%,11.9%,-4.3%,+11.7%
4,"8,759",6.3%,0.3773,0.4386,5.7%,11.3%,-1.3%,+14.0%
5,"8,759",6.3%,0.3844,0.4421,5.9%,11.2%,-5.4%,+13.1%
6,"8,727",6.2%,0.3910,0.4501,6.1%,11.3%,-4.3%,+13.1%
7,"8,727",6.2%,0.3954,0.4637,6.3%,11.8%,-5.8%,+14.7%
8,"8,727",6.2%,0.4042,0.4917,6.6%,12.1%,-5.7%,+17.8%
9,"8,727",6.2%,0.4070,0.4905,6.6%,13.4%,-7.1%,+17.0%


### 11.2 By store and by product family

In [19]:
S = seg("store_nbr").sort_values("RMSLE model"); F = seg("family").sort_values("RMSLE model")
fig, axs = plt.subplots(1, 2, figsize=(12, 6.2))
top = F.sort_values("share of model SSE", ascending=False).head(12).iloc[::-1]
axs[0].barh(top.index, top["share of model SSE"], color=BLUE, height=0.65); axs[0].set_xlabel("share of total squared error"); axs[0].set_title("12 families that carry most of the error", loc="left", fontsize=10)
for y, (v, r) in enumerate(zip(top["share of model SSE"], top["RMSLE model"])): axs[0].text(v + 0.002, y, f"RMSLE {r:.2f}", va="center", fontsize=7.5, color=INK2)
sw = pd.concat([S.head(6), S.tail(6)]); sw.index = sw.index.astype(str)
axs[1].barh("store " + sw.index[::-1], sw["RMSLE model"][::-1], color=[ORANGE] * 6 + [AQUA] * 6, height=0.65); axs[1].axvline(np.sqrt(VAL.se.mean()), color=INK2, lw=0.8, ls="--")
axs[1].set_xlabel("RMSLE"); axs[1].set_title("6 hardest (orange) and 6 easiest (green) stores", loc="left", fontsize=10)
ftitle(fig, "Where the error sits - product families and stores"); finish(fig, "p4_error_store_family")
print(f"store RMSLE range {S['RMSLE model'].min():.3f} - {S['RMSLE model'].max():.3f} | family RMSLE range {F['RMSLE model'].min():.3f} - {F['RMSLE model'].max():.3f}")
F.sort_values("share of model SSE", ascending=False).head(12).style.format(FMT)

store RMSLE range 0.307 - 0.465 | family RMSLE range 0.130 - 0.658


,rows,row share,RMSLE model,RMSLE baseline,share of model SSE,WAPE,bias,gain vs baseline
family,,,,,,,,
SCHOOL AND OFFICE SUPPLIES,"4,261",3.0%,0.6583,1.1032,8.5%,73.2%,-62.9%,+40.3%
LINGERIE,"4,261",3.0%,0.6262,0.7135,7.7%,48.6%,-18.8%,+12.2%
GROCERY II,"4,261",3.0%,0.5994,0.6559,7.0%,34.9%,-18.3%,+8.6%
CELEBRATION,"4,261",3.0%,0.5454,0.6044,5.8%,37.9%,-13.0%,+9.8%
HARDWARE,"4,261",3.0%,0.5351,0.6060,5.6%,73.9%,-24.7%,+11.7%
AUTOMOTIVE,"4,261",3.0%,0.5020,0.5592,4.9%,38.0%,-12.5%,+10.2%
MAGAZINES,"4,261",3.0%,0.4973,0.5473,4.8%,37.4%,-17.6%,+9.1%
SEAFOOD,"4,261",3.0%,0.4889,0.5589,4.7%,22.8%,-0.5%,+12.5%
"LIQUOR,WINE,BEER","4,261",3.0%,0.4864,0.5657,4.6%,27.8%,-12.5%,+14.0%


In [20]:
S.tail(8).style.format(FMT)

,rows,row share,RMSLE model,RMSLE baseline,share of model SSE,WAPE,bias,gain vs baseline
store_nbr,,,,,,,,
43,"2,624",1.9%,0.4317,0.4847,2.2%,12.7%,-2.4%,+10.9%
36,"2,624",1.9%,0.4319,0.5183,2.3%,15.8%,-5.4%,+16.7%
13,"2,624",1.9%,0.4368,0.5150,2.3%,15.1%,-3.9%,+15.2%
19,"2,624",1.9%,0.4434,0.4866,2.4%,11.7%,-1.3%,+8.9%
32,"2,624",1.9%,0.4447,0.5488,2.4%,34.4%,-16.9%,+19.0%
30,"2,624",1.9%,0.4464,0.5554,2.4%,24.4%,-4.6%,+19.6%
22,"2,624",1.9%,0.4509,0.4998,2.5%,16.9%,-5.5%,+9.8%
26,"2,624",1.9%,0.4645,0.5460,2.6%,19.6%,-1.3%,+14.9%


### 11.3 By volume, activity type, promotion, weekday and holiday

In [21]:
V = seg("volume", ["no history", "<=1 unit/day", "1-5", "5-20", "20-100", ">100"]); A = seg("activity"); P = seg("promo"); W = seg("weekday", ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]); HD = seg("holiday")
segs = pd.concat({"volume (28-day mean units/day)": V, "activity type": A, "promotion": P, "weekday": W, "holiday": HD}, names=["dimension", "segment"])
fig, axs = plt.subplots(1, 5, figsize=(14, 3.6), sharey=True)
for ax, (nm_, t) in zip(axs, [("volume", V), ("activity", A), ("promotion", P), ("weekday", W), ("holiday", HD)]):
    x = np.arange(len(t)); ax.bar(x - 0.2, t["RMSLE baseline"], 0.38, color=NEUTRAL); ax.bar(x + 0.2, t["RMSLE model"], 0.38, color=BLUE)
    ax.set_xticks(x); ax.set_xticklabels([str(s).replace(" (rule_zero)", "").replace(" unit/day", "").replace("holiday / event day", "holiday").replace("on promotion", "promo").replace("no promotion", "no promo").replace("regular day", "regular") for s in t.index], rotation=35, ha="right", fontsize=8)
    ax.set_title(nm_, loc="left", fontsize=10)
axs[0].set_ylabel("RMSLE (grey = baseline, blue = model)"); ftitle(fig, "RMSLE by segment"); finish(fig, "p4_error_segments")
segs.style.format(FMT)

### 11.4 Best and worst segments (families x horizon band)

In [22]:
VAL["h_band"] = pd.cut(VAL.horizon, [0, 4, 8, 12, 16], labels=["h1-4", "h5-8", "h9-12", "h13-16"])
fh = VAL.groupby(["family", "h_band"], observed=True).se.mean().pow(0.5).unstack()
BEST_FAM = F.head(3).index.tolist(); WORST_FAM = F.tail(3).index.tolist()
print("easiest families:", ", ".join(f"{i} ({v:.3f})" for i, v in F["RMSLE model"].head(3).items()))
print("hardest families:", ", ".join(f"{i} ({v:.3f})" for i, v in F["RMSLE model"].tail(3).items()))
fh.loc[BEST_FAM + WORST_FAM].style.format("{:.3f}")

easiest families: PRODUCE (0.130), DAIRY (0.134), BREAD/BAKERY (0.154)
hardest families: GROCERY II (0.599), LINGERIE (0.626), SCHOOL AND OFFICE SUPPLIES (0.658)


h_band,h1-4,h5-8,h9-12,h13-16
family,,,,
PRODUCE,0.128,0.116,0.145,0.128
DAIRY,0.128,0.119,0.147,0.139
BREAD/BAKERY,0.148,0.140,0.158,0.169
GROCERY II,0.613,0.588,0.581,0.615
LINGERIE,0.609,0.639,0.627,0.630
SCHOOL AND OFFICE SUPPLIES,0.559,0.661,0.741,0.660


## 12. Feature importance
Two complementary views: **gain** (how much each feature reduced the training loss inside the trees, averaged over the five fold models) and **SHAP** (how much each feature moved individual forecasts, on a sample of the latest fold). Both describe *what the model uses to predict* - not what *causes* sales. A feature that ranks high may simply be a proxy (e.g. the 4-week same-weekday mean is a proxy for the store-family's current level).

In [23]:
IMP_SRC = LGB_FINAL                     # LightGBM is always trained with native gain importance, whatever model won
GS = gain_share(IMP_SRC)
grp = fdict.set_index("feature").group
IMP = pd.DataFrame({"gain share": GS, "group": grp[GS.index], "rank": range(1, len(GS) + 1)})
IMP["cumulative"] = IMP["gain share"].cumsum()
top20 = IMP.head(20)
gcol = {g: c for g, c in zip(sorted(top20.group.unique()), [BLUE, ORANGE, AQUA, "#7a5cc9", "#d1a300", "#c3c2b7", "#e05a8c", "#3a9ac8", "#8a8a7a"])}
fig, ax = plt.subplots(figsize=(9.5, 6.2))
ax.barh(top20.index[::-1], top20["gain share"][::-1], color=[gcol[g] for g in top20.group[::-1]], height=0.7)
for y, v in enumerate(top20["gain share"][::-1]): ax.text(v + 0.002, y, f"{v:.1%}", va="center", fontsize=8, color=INK2)
ax.set_xlabel("share of total gain (mean of 5 folds)"); ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in gcol.values()], labels=[g[3:] for g in gcol], frameon=False, fontsize=8, loc="lower right", title="feature group", title_fontsize=8.5)
ftitle(fig, "Top-20 features by gain (LightGBM, selected feature set)"); finish(fig, "p4_importance_gain")
grp_share = IMP.groupby("group")["gain share"].sum().sort_values(ascending=False)
print(f"top-5 features carry {IMP['gain share'].head(5).sum():.1%} of gain, top-20 {IMP['gain share'].head(20).sum():.1%}; {int((IMP['gain share'] < 0.001).sum())} of {len(IMP)} features each carry < 0.1%")
grp_share.to_frame("gain share").style.format("{:.1%}")

top-5 features carry 95.3% of gain, top-20 99.0%; 63 of 78 features each carry < 0.1%


,gain share
group,
F. Horizon-aligned same-weekday lags,86.4%
G. Rolling level,11.3%
I. Growth & year-ago,0.5%
J. Activity & intermittency,0.4%
E. Origin-relative lags,0.4%
A. Horizon & calendar,0.3%
B. Store & product identity,0.3%
D. Promotions,0.2%
H. Volatility,0.1%


In [24]:
def shap_job():
    r = fc.run_lgb(mt, "F5_latest", FEAT_FINAL, FEAT_PARAMS, FEAT_ROUNDS, categorical=LGB_CAT, return_model=True)
    _, va_idx, va_mod = fc.fold_frames(mt, "F5_latest")
    rng = np.random.default_rng(0); take = np.sort(rng.choice(va_mod, size=20000, replace=False))
    X = mt.iloc[take][FEAT_FINAL]; sv = r["model"].predict(X, pred_contrib=True)
    return dict(X=X.reset_index(drop=True), sv=sv[:, :-1].astype(np.float32), base_value=float(sv[0, -1]))
SH = fc.cached("shap_F5", shap_job)
mean_abs = pd.Series(np.abs(SH["sv"]).mean(0), index=SH["X"].columns).sort_values(ascending=False)
try:
    import shap
    plt.figure(figsize=(9, 6.5)); shap.summary_plot(SH["sv"], SH["X"], max_display=15, show=False, plot_size=None); plt.title("SHAP summary - latest fold (F5), 20,000 modeled validation rows", loc="left", fontsize=11, fontweight="bold")
    plt.tight_layout(); plt.savefig("figs/p4_shap.png", dpi=110); plt.show()
except Exception as e:
    print("SHAP plot skipped:", e)
cmp_imp = pd.DataFrame({"gain rank": IMP["rank"].reindex(mean_abs.index), "SHAP rank": range(1, len(mean_abs) + 1), "mean |SHAP| (log units)": mean_abs.values}, index=mean_abs.index).head(15)
cmp_imp.style.format({"gain rank": "{:.0f}", "mean |SHAP| (log units)": "{:.4f}"})

,gain rank,SHAP rank,mean |SHAP| (log units)
sd_mean_8,1,1,0.9853
sd_mean_4,2,2,0.4969
roll_mean_7d,3,3,0.1772
yoy_lag_364,10,4,0.0971
roll_mean_14d,5,5,0.0902
lag_1d,8,6,0.0795
sd_lag_1,4,7,0.0582
yoy_mean_15d,14,8,0.0572
roll_median_28d,6,9,0.0479
promo_ratio_28d,21,10,0.0386


## 13. The `rule_zero` decision - evaluated separately
Phase 3 decided that series that have not sold for a long time (dormant) or never sold are forecast as **0 units** and excluded from training. That is a *modeling shortcut*, not a claim about stock: a zero-sales series can be out of stock, delisted, seasonal or simply not carried in that store, and sales data alone cannot tell which. Here we only test whether the shortcut is **numerically sound**: how many rows it touches, how often 0 is the right answer, and what it does to RMSLE compared with letting the model forecast those rows.

In [25]:
RZ_RES = LGB_FINAL                       # single model with `pred_norule` stored for every validation row
rows = []
for r in RZ_RES:
    i = r["va_idx"]; d = mt.iloc[i]; rz = d.rule_zero.to_numpy() == 1; y = d.y_log1p.to_numpy(); yu = np.expm1(y)
    n = int(rz.sum()); tot = len(i)
    sse_rule = float(((r["pred"][rz] - y[rz]) ** 2).sum()); sse_model = float(((r["pred_norule"][rz] - y[rz]) ** 2).sum()); sse_rest = float(((r["pred"][~rz] - y[~rz]) ** 2).sum())
    rows.append({"fold": r["fold"].split("_")[0], "rule_zero rows": n, "share of rows": n / tot, "actual sales = 0": float((yu[rz] == 0).mean()), "mean actual units": float(yu[rz].mean()), "units sold on these rows": float(yu[rz].sum()),
                 "share of SSE (rule)": sse_rule / (sse_rule + sse_rest), "share of SSE (model instead)": sse_model / (sse_model + sse_rest), "RMSLE with rule": r["rmsle"], "RMSLE model only": r["rmsle_norule"]})
RZ = pd.DataFrame(rows).set_index("fold")
RZ.loc["mean"] = RZ.mean()
print(f"pooled: rule_zero rows {int(RZ['rule_zero rows'].iloc[:-1].sum()):,}; forecasting them as 0 lowers mean RMSLE from {RZ.loc['mean', 'RMSLE model only']:.4f} to {RZ.loc['mean', 'RMSLE with rule']:.4f}")
RZ.style.format({"rule_zero rows": "{:,.0f}", "share of rows": "{:.1%}", "actual sales = 0": "{:.1%}", "mean actual units": "{:.3f}", "units sold on these rows": "{:,.1f}", "share of SSE (rule)": "{:.2%}", "share of SSE (model instead)": "{:.2%}", "RMSLE with rule": "{:.4f}", "RMSLE model only": "{:.4f}"})

pooled: rule_zero rows 5,690; forecasting them as 0 lowers mean RMSLE from 0.3964 to 0.3961


,rule_zero rows,share of rows,actual sales = 0,mean actual units,units sold on these rows,share of SSE (rule),share of SSE (model instead),RMSLE with rule,RMSLE model only
fold,,,,,,,,,
F1,842,3.2%,99.5%,0.008,7.0,0.10%,0.27%,0.3874,0.3877
F2,"1,168",4.2%,99.4%,0.009,11.0,0.13%,0.35%,0.4159,0.4163
F3,"1,168",4.1%,100.0%,0.000,0.0,0.00%,0.18%,0.3834,0.3838
F4,"1,248",4.4%,100.0%,0.000,0.0,0.00%,0.12%,0.3882,0.3884
F5,"1,264",4.4%,99.4%,0.009,12.0,0.15%,0.32%,0.4055,0.4058
mean,"1,138",4.1%,99.7%,0.005,6.0,0.08%,0.25%,0.3961,0.3964


In [26]:
rzv = VAL[VAL.rule_zero == 1]
wrong = rzv[rzv.y_u > 0]
print(f"rows where the rule forecasts 0 but something sold: {len(wrong):,} of {len(rzv):,} ({len(wrong) / len(rzv):.1%}); mean units sold in those rows {wrong.y_u.mean():.2f}; max {wrong.y_u.max():.0f}")
print("series (store x family) with at least one such sale:", wrong.groupby(["store_nbr", "family"]).ngroups, "of", rzv.groupby(["store_nbr", "family"]).ngroups)
print(f"at the real origin (2017-08-15): {int(tf.rule_zero.sum() // 16)} series = {int(tf.rule_zero.sum()):,} of {len(tf):,} forecast rows ({tf.rule_zero.mean():.1%}) will be forecast as 0")

rows where the rule forecasts 0 but something sold: 18 of 5,690 (0.3%); mean units sold in those rows 1.67; max 3
series (store x family) with at least one such sale: 5 of 98
at the real origin (2017-08-15): 80 series = 1,280 of 28,512 forecast rows (4.5%) will be forecast as 0


## 14. Final model selection
The selection uses **all five folds** and several angles at once - mean RMSLE, the worst fold, the two most recent folds, the seasonal-twin fold F1, the spread across folds and cost - rather than one lucky window.

In [27]:
sel = CMP.drop(index="Baseline (mean of last 4 same weekdays)").sort_values("Mean RMSLE")
sel = sel[["Mean RMSLE", "Improvement vs Baseline", "Worst fold", "F4+F5 mean", "F1", "Std", "Folds beating baseline", "Train s/fold"]]
sel["selected"] = np.where(sel.index == WINNER, "<-- SELECTED", "")
print("Selected:", WINNER)
sel.style.format({"Mean RMSLE": "{:.4f}", "Improvement vs Baseline": "{:.1%}", "Worst fold": "{:.4f}", "F4+F5 mean": "{:.4f}", "F1": "{:.4f}", "Std": "{:.4f}", "Train s/fold": "{:.0f}"})

Selected: Blend 50/50: LightGBM (selected) + XGBoost


,Mean RMSLE,Improvement vs Baseline,Worst fold,F4+F5 mean,F1,Std,Folds beating baseline,Train s/fold,selected
Model,,,,,,,,,
Blend 50/50: LightGBM (selected) + XGBoost,0.3942,16.0%,0.4139,0.3954,0.3852,0.0124,5,157,<-- SELECTED
"XGBoost, tier 1+2",0.3959,15.6%,0.4162,0.3966,0.3872,0.0124,5,112,
"LightGBM, residual target",0.3960,15.6%,0.4123,0.3959,0.3912,0.0101,5,45,
"LightGBM, tier 1+2",0.3961,15.6%,0.4159,0.3968,0.3874,0.0125,5,45,
"XGBoost, tier 1",0.3979,15.2%,0.4155,0.3987,0.3910,0.0103,5,61,
"LightGBM, tier 1",0.3980,15.2%,0.4171,0.3999,0.3890,0.0119,5,27,
"Random Forest, tier 1+2",0.4058,13.5%,0.4297,0.4104,0.3954,0.0160,5,158,
"Random Forest, tier 1",0.4076,13.1%,0.4346,0.4140,0.3972,0.0164,5,50,


In [28]:
def feat_label(f):
    f = set(f)
    return "tier 1" if f == set(T1) else "tier 1+2" if f == set(T12) else "tier 1+2 pruned" if f < set(T12) else "tier 1+2 + tier-3 group(s): " + ", ".join(sorted(f - set(T12)))
wr = np.array([r["rmsle"] for r in WIN_RES]); br = np.array([r["rmsle"] for r in BASE])
SEL_DOC = pd.DataFrame({"item": ["algorithm", "feature set", "number of features", "categorical features", "hyper-parameters", "boosting rounds", "target", "training rows in a fold (mean)", "validation RMSLE per fold (F1..F5)", "validation RMSLE mean",
                                 "baseline RMSLE mean", "improvement vs baseline (mean)", "improvement vs baseline per fold", "worst fold", "training time per fold", "inference time per fold"],
                        "value": [WINNER, " | ".join(feat_label(c["features"]) for c in WIN_SPEC), ", ".join(str(len(c["features"])) for c in WIN_SPEC), ", ".join(sorted(set(sum([c["kw"].get("categorical", []) for c in WIN_SPEC], [])))) or "none",
                                  " | ".join(json.dumps(c["params"]) for c in WIN_SPEC), " | ".join(str(c["rounds"]) for c in WIN_SPEC), "y_log1p = log(1 + sales)" + (" - residual to the baseline" if any(c["kw"].get("target") == "residual" for c in WIN_SPEC) else ""),
                                  f"{np.mean([r.get('n_train', np.nan) for r in LGB_FINAL]):,.0f}", " / ".join(f"{v:.4f}" for v in wr), f"{wr.mean():.4f}", f"{br.mean():.4f}", f"{1 - wr.mean() / br.mean():.1%}",
                                  " / ".join(f"{1 - a / b:.1%}" for a, b in zip(wr, br)), f"{wr.max():.4f} ({WIN_RES[int(wr.argmax())]['fold']})", f"{np.mean([r['train_s'] for r in WIN_RES]):.0f} s", f"{np.mean([r['infer_s'] for r in WIN_RES]):.1f} s"]}).set_index("item")
SEL_DOC

,value
item,
algorithm,Blend 50/50: LightGBM (selected) + XGBoost
feature set,tier 1+2 | tier 1+2
number of features,"78, 78"
categorical features,"city_id, dow, event, family_id, hol_type, stat..."
hyper-parameters,"{""objective"": ""regression"", ""num_threads"": 2, ..."
boosting rounds,173 | 300
target,y_log1p = log(1 + sales)
training rows in a fold (mean),"2,323,275"
validation RMSLE per fold (F1..F5),0.3852 / 0.4139 / 0.3812 / 0.3874 / 0.4034


## 15. Final refit and the 16-day forecast
The selected configuration is refitted on **every eligible row known at the real forecast origin (2017-08-15)** - all forecast origins from 2015-07-01 whose targets are already observed - with `rule_zero` rows excluded, exactly as in validation. The boosting rounds are the ones fixed from the validation folds (they are not re-tuned). Boosting models are averaged over 2 random seeds to reduce the variance of the row/feature sampling. Predictions are clipped at 0 in log space, converted with `expm1` and **not rounded**; `rule_zero` series are forecast as exactly 0.

In [29]:
import lightgbm as lgb, xgboost as xgb, joblib, pickle
tr_mask = ((mt.origin_date >= pd.Timestamp("2015-07-01")) & (mt.rule_zero == 0)).to_numpy()
assert mt.target_date.max() <= pd.Timestamp("2017-08-15") and mt.origin_date.max() < pd.Timestamp("2017-08-15"), "training targets must be known at the forecast origin"
FINAL_TR = np.flatnonzero(tr_mask)
print(f"final training set: {len(FINAL_TR):,} rows | origins {mt.origin_date.iloc[FINAL_TR].min().date()} -> {mt.origin_date.iloc[FINAL_TR].max().date()} | latest training target {mt.target_date.iloc[FINAL_TR].max().date()}")
for c in set(sum([s["features"] for s in WIN_SPEC], [])): assert c in tf.columns and mt[c].dtype == tf[c].dtype, c
TEST_BASE = fc.baseline_log(tf)
SEEDS = [0, 1]                              # 2 seeds per component (memory-constrained container); results are cached per (component, seed)
import gc
MD = "phase4/outputs/models"; os.makedirs(MD, exist_ok=True)
MODEL_EXT = {"lgb": "txt", "xgb": "json", "rf": "joblib"}
def fit_one(kind, feats, params, rounds, cat, resid, s, model_path):
    """Fit one final component/seed on the full refit window, predict the 28,512 test rows and save the model straight to disk
    (never held in memory afterwards - the container is memory-constrained). Cached (pred + timing) so a container restart
    resumes at the next (component, seed) instead of redoing the whole refit."""
    d = mt.iloc[FINAL_TR]
    y = d.y_log1p.to_numpy(np.float64) - (fc.residual_base(d) if resid else 0)
    t0 = time.time()
    if kind == "lgb":
        ds = lgb.Dataset(d[feats].astype(np.float32), y, categorical_feature=cat)
        m = lgb.train(dict(params, seed=s, verbose=-1), ds, num_boost_round=rounds)
        p = m.predict(tf[feats].astype(np.float32)); m.save_model(model_path); del ds
    elif kind == "xgb":
        dtr = xgb.DMatrix(d[feats].to_numpy(np.float32), y, feature_names=list(feats))
        m = xgb.train(dict(params, seed=s), dtr, num_boost_round=rounds)
        p = m.predict(xgb.DMatrix(tf[feats].to_numpy(np.float32), feature_names=list(feats))); m.save_model(model_path); del dtr
    else:
        from sklearn.ensemble import RandomForestRegressor
        sub = np.sort(np.random.default_rng(s).choice(len(d), 250_000, replace=False))
        m = RandomForestRegressor(random_state=s, n_jobs=2, **params).fit(d.iloc[sub][feats].fillna(-1).to_numpy(np.float32), y[sub])
        p = m.predict(tf[feats].fillna(-1).to_numpy(np.float32)); joblib.dump(m, model_path, compress=3)
    pred = (p + (TEST_BASE if resid else 0)).astype(np.float32)
    train_s = time.time() - t0
    del d, y, m; gc.collect()
    return pred, train_s
FINAL_MODELS = []; comp_preds = []; timing = []
for ci, c in enumerate(WIN_SPEC):
    feats = c["features"]; resid = c["kw"].get("target") == "residual"; cat = [x for x in c["kw"].get("categorical", []) if x in feats]
    preds = []
    for s in (SEEDS if c["kind"] != "rf" else [0]):
        key = f"final_{c['kind']}_c{ci}_s{s}"; path = f"{fc.CACHE_DIR}/{key}.pkl"
        model_path = f"{MD}/final_{c['kind']}_component{ci}_seed{s}.{MODEL_EXT[c['kind']]}"
        if os.path.exists(path) and os.path.exists(model_path):
            res = pickle.load(open(path, "rb")); pred, train_s, from_cache = res["pred"], res["train_s"], True
        else:
            pred, train_s = fit_one(c["kind"], feats, c["params"], c["rounds"], cat, resid, s, model_path)
            pickle.dump({"pred": pred, "train_s": train_s}, open(path, "wb")); from_cache = False
        preds.append(pred.astype(np.float64))
        FINAL_MODELS.append((c["kind"], ci, s, model_path)); timing.append(train_s)
        print(f"  component {ci} ({c['kind']}) seed {s}: {'loaded from cache' if from_cache else f'fitted in {train_s:.0f}s'}")
        gc.collect()
    comp_preds.append(np.mean(preds, axis=0))
raw_log = np.mean(comp_preds, axis=0)
TEST_LOG = np.where(tf.rule_zero.to_numpy() == 1, 0.0, np.clip(raw_log, 0, None))
TEST_UNITS = np.expm1(TEST_LOG)
print(f"forecast rows: {len(TEST_UNITS):,} | zeros from rule_zero: {int((tf.rule_zero == 1).sum()):,} | total units {TEST_UNITS.sum():,.0f} | mean per row {TEST_UNITS.mean():.2f}")

final training set: 2,839,036 rows | origins 2015-07-07 -> 2017-08-08 | latest training target 2017-08-15
  component 0 (lgb) seed 0: loaded from cache
  component 0 (lgb) seed 1: loaded from cache


  component 1 (xgb) seed 0: loaded from cache
  component 1 (xgb) seed 1: loaded from cache
forecast rows: 28,512 | zeros from rule_zero: 1,280 | total units 12,563,212 | mean per row 440.63


In [30]:
# sanity checks of the forecast: agrees with the baseline in level, has sensible weekday/horizon shape
chk2 = pd.DataFrame({"date": tf.target_date.to_numpy(), "h": tf.horizon.to_numpy(), "model": TEST_UNITS, "baseline": np.expm1(TEST_BASE)})
day = chk2.groupby("date")[["model", "baseline"]].sum(); day["ratio"] = day.model / day.baseline
fig, ax = plt.subplots(figsize=(9.5, 3.8))
ax.plot(day.index, day.baseline, color=NEUTRAL, lw=2, marker="o", ms=4, label="baseline (mean of last 4 same weekdays)"); ax.plot(day.index, day.model, color=BLUE, lw=2, marker="o", ms=4, label="selected model")
ax.set_ylabel("units, all stores and families"); ax.legend(frameon=False); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1e3:,.0f}k"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right"); ftitle(fig, "Forecast 2017-08-16 -> 2017-08-31: total daily units - model vs baseline"); finish(fig, "p4_forecast_total")
print(f"model total = {day.model.sum():,.0f} units vs baseline {day.baseline.sum():,.0f} ({day.ratio.mean() - 1:+.1%} on average); daily ratio range {day.ratio.min():.3f} - {day.ratio.max():.3f}")
day.round(1).style.format({"model": "{:,.0f}", "baseline": "{:,.0f}", "ratio": "{:.3f}"})

model total = 12,563,212 units vs baseline 13,202,203 (-4.9% on average); daily ratio range 0.870 - 1.036


,model,baseline,ratio
date,,,
2017-08-16 00:00:00,"816,014","801,871",1.000
2017-08-17 00:00:00,"657,124","681,898",1.000
2017-08-18 00:00:00,"762,530","817,917",0.900
2017-08-19 00:00:00,"911,876","930,884",1.000
2017-08-20 00:00:00,"1,021,577","1,015,810",1.000
2017-08-21 00:00:00,"787,051","815,202",1.000
2017-08-22 00:00:00,"727,256","795,634",0.900
2017-08-23 00:00:00,"748,027","801,871",0.900
2017-08-24 00:00:00,"634,045","681,898",0.900


## 16. Output files
`final_forecast.parquet` (business view, one row per store x family x day) and `submission.csv` (Kaggle format, `id, sales`, in the order of `sample_submission.csv`).

In [31]:
OUT = "phase4/outputs"; os.makedirs(OUT, exist_ok=True)
FAMN = FAM.to_dict()
ff = pd.DataFrame({"id": tf.id.to_numpy(), "date": pd.to_datetime(tf.target_date.to_numpy()), "store_nbr": tf.store_nbr.to_numpy(), "family": tf.family_id.map(FAMN).to_numpy(), "predicted_sales": TEST_UNITS,
                   "onpromotion": tf.onpromotion.to_numpy(), "forecast_horizon": tf.horizon.to_numpy()})
ff = ff.merge(activity[["store_nbr", "family", "status"]].rename(columns={"status": "activity_status"}), on=["store_nbr", "family"], how="left", validate="m:1")
ff = ff.sort_values("id").reset_index(drop=True)
sp = f"data/sample_submission.csv"
sample = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame({"id": np.sort(tf.id.to_numpy()), "sales": 0.0})
sub = sample[["id"]].merge(ff[["id", "predicted_sales"]].rename(columns={"predicted_sales": "sales"}), on="id", how="left", validate="1:1")
ff[["date", "store_nbr", "family", "predicted_sales", "onpromotion", "forecast_horizon", "activity_status"]].to_parquet(f"{OUT}/final_forecast.parquet", index=False)
sub.to_csv(f"{OUT}/submission.csv", index=False)
ffc = pd.read_parquet(f"{OUT}/final_forecast.parquet"); subc = pd.read_csv(f"{OUT}/submission.csv")
VER = [("final_forecast rows = 54 stores x 33 families x 16 days = 28,512", len(ffc), len(ffc) == 28_512),
       ("submission rows = 28,512 and equal to sample_submission", len(subc), len(subc) == 28_512 and len(sample) == 28_512),
       ("submission ids unique and identical (order included) to sample_submission", int(subc.id.is_unique), subc.id.is_unique and (subc.id.to_numpy() == sample.id.to_numpy()).all()),
       ("no missing values (forecast and submission)", int(ffc.drop(columns="activity_status").isna().sum().sum() + subc.isna().sum().sum()), ffc.drop(columns="activity_status").isna().sum().sum() + subc.isna().sum().sum() == 0),
       ("no negative predictions", f"min {min(ffc.predicted_sales.min(), subc.sales.min()):.4f}", ffc.predicted_sales.min() >= 0 and subc.sales.min() >= 0),
       ("dates 2017-08-16 -> 2017-08-31, horizons 1..16", f"{ffc.date.min().date()} -> {ffc.date.max().date()}", ffc.date.min() == pd.Timestamp("2017-08-16") and ffc.date.max() == pd.Timestamp("2017-08-31") and sorted(ffc.forecast_horizon.unique()) == list(range(1, 17))),
       ("1,782 store-family series per day; 54 stores; 33 families", f"{ffc.groupby('date').size().unique().tolist()} | {ffc.store_nbr.nunique()} | {ffc.family.nunique()}", ffc.groupby("date").size().eq(1782).all() and ffc.store_nbr.nunique() == 54 and ffc.family.nunique() == 33),
       ("submission equals forecast (same numbers, not rounded)", "max abs diff %.2e" % np.abs(sub.merge(ff[["id", "predicted_sales"]], on="id").eval("sales - predicted_sales")).max(), np.allclose(sub.merge(ff[["id", "predicted_sales"]], on="id").eval("sales - predicted_sales"), 0)),
       ("rule_zero series forecast as exactly 0", int(((tf.rule_zero == 1).to_numpy() & (TEST_UNITS != 0)).sum()), not ((tf.rule_zero == 1).to_numpy() & (TEST_UNITS != 0)).any()),
       ("activity_status filled for every row", int(ffc.activity_status.isna().sum()), ffc.activity_status.notna().all()),
       ("modeling table unchanged since section 1 (fingerprint)", str(FINGERPRINT[1:]), FINGERPRINT == ((mt.shape, round(float(mt.y_log1p.sum()), 3), round(float(mt.sd_mean_4.sum()), 3), int(mt.rule_zero.sum()))))]
VER = pd.DataFrame(VER, columns=["check", "result", "ok"]); assert VER.ok.all(), VER[~VER.ok]
print("all output checks passed"); VER.drop(columns="ok")

all output checks passed


,check,result
0,final_forecast rows = 54 stores x 33 families ...,28512
1,"submission rows = 28,512 and equal to sample_s...",28512
2,submission ids unique and identical (order inc...,1
3,no missing values (forecast and submission),0
4,no negative predictions,min 0.0000
5,"dates 2017-08-16 -> 2017-08-31, horizons 1..16",2017-08-16 -> 2017-08-31
6,"1,782 store-family series per day; 54 stores; ...",[1782] | 54 | 33
7,"submission equals forecast (same numbers, not ...",max abs diff 0.00e+00
8,rule_zero series forecast as exactly 0,0
9,activity_status filled for every row,0


In [32]:
ffc.head(8)

,date,store_nbr,family,predicted_sales,onpromotion,forecast_horizon,activity_status
0,2017-08-16,1,AUTOMOTIVE,4.2248,0,1,regular
1,2017-08-16,1,BABY CARE,0.0000,0,1,never sold
2,2017-08-16,1,BEAUTY,5.0565,2,1,regular
3,2017-08-16,1,BEVERAGES,"2,520.9931",20,1,regular
4,2017-08-16,1,BOOKS,0.0888,0,1,intermittent
5,2017-08-16,1,BREAD/BAKERY,424.0263,12,1,regular
6,2017-08-16,1,CELEBRATION,13.1801,0,1,regular
7,2017-08-16,1,CLEANING,824.8969,25,1,regular


In [33]:
by = ffc.groupby("activity_status").agg(series=("family", lambda s: len(s) // 16), units=("predicted_sales", "sum"))
by["share of units"] = by.units / by.units.sum()
fam_tot = ffc.groupby("family").predicted_sales.sum().sort_values(ascending=False)
print("forecast units by family - top 5:", ", ".join(f"{k} {v / fam_tot.sum():.1%}" for k, v in fam_tot.head(5).items()))
by.style.format({"units": "{:,.0f}", "share of units": "{:.2%}"})

forecast units by family - top 5: GROCERY I 29.4%, BEVERAGES 22.7%, PRODUCE 15.6%, CLEANING 7.7%, DAIRY 5.6%


,series,units,share of units
activity_status,,,
dormant,27,0,0.00%
intermittent,158,"2,898",0.02%
never sold,53,0,0.00%
regular,1544,"12,560,315",99.98%


## 17. Saved artifacts
Everything needed to audit, reload or reproduce the result: model files, feature list, hyper-parameters, fold scores, comparison and ablation tables, importance, validation predictions and test predictions.

In [34]:
import platform, sklearn, glob
saved = [model_path for _, _, _, model_path in FINAL_MODELS]                 # models were saved straight to disk when fitted (section 15)
assert all(os.path.exists(p) for p in saved), "a final model file is missing"
final_cfg = {"selected_model": WINNER, "components": [dict(kind=c["kind"], n_features=len(c["features"]), features=c["features"], params=c["params"], rounds=c["rounds"], options=c["kw"]) for c in WIN_SPEC], "seeds": SEEDS,
             "target": "y_log1p = log1p(sales); predictions clipped at 0 then expm1, not rounded", "rule_zero": "rows with rule_zero==1 excluded from training and forecast as 0",
             "training_rows": int(len(FINAL_TR)), "training_origins": [str(mt.origin_date.iloc[FINAL_TR].min().date()), str(mt.origin_date.iloc[FINAL_TR].max().date())], "forecast_origin": "2017-08-15", "horizons": "1..16",
             "validation": {"metric": "RMSLE", "folds": {r["fold"]: round(r["rmsle"], 5) for r in WIN_RES}, "mean": round(float(wr.mean()), 5), "baseline_mean": round(float(br.mean()), 5), "improvement_vs_baseline": round(float(1 - wr.mean() / br.mean()), 4)},
             "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "lightgbm": lgb.__version__, "xgboost": xgb.__version__, "sklearn": sklearn.__version__}}
json.dump(final_cfg, open(f"{OUT}/final_model_config.json", "w"), indent=2, default=str)
json.dump({"features": sum([c["features"] for c in WIN_SPEC], []) if len(WIN_SPEC) == 1 else {f"component_{i}": c["features"] for i, c in enumerate(WIN_SPEC)}}, open(f"{OUT}/feature_list.json", "w"), indent=2)
json.dump({"lightgbm_grid": LGB_GRID, "lightgbm_selected": LGB_BEST, "xgboost_grid": XGB_GRID, "xgboost_selected": XGB_BEST, "random_forest_grid": RF_GRID, "random_forest_selected": RF_BEST, "rounds": {"lgb_t1": ROUNDS_T1, "lgb_t12": ROUNDS_T12, "xgb": XGB_ROUNDS}}, open(f"{OUT}/hyperparameters_tried.json", "w"), indent=2, default=str)
fold_scores = pd.DataFrame({k: {r["fold"]: r["rmsle"] for r in v} for k, v in CANDS.items()}).T; fold_scores.to_csv(f"{OUT}/fold_scores.csv")
CMP.to_csv(f"{OUT}/model_comparison.csv"); abl.to_csv(f"{OUT}/feature_ablation.csv"); IMP.to_csv(f"{OUT}/feature_importance_gain.csv"); mean_abs.rename("mean_abs_shap").to_csv(f"{OUT}/feature_importance_shap_F5.csv")
segs.to_csv(f"{OUT}/error_by_segment.csv"); pd.concat({"store": S, "family": F}, names=["dimension", "segment"]).to_csv(f"{OUT}/error_by_store_family.csv"); RZ.to_csv(f"{OUT}/rule_zero_evaluation.csv")
VAL.drop(columns=["h_band", "volume", "promo", "weekday", "holiday", "activity"]).to_parquet(f"{OUT}/validation_predictions.parquet", index=False)
tp = ffc.copy(); tp.insert(0, "id", ff.id.to_numpy()); tp["pred_log1p"] = TEST_LOG; tp["baseline_units"] = np.expm1(TEST_BASE); tp.to_parquet(f"{OUT}/test_predictions.parquet", index=False)
sz = pd.DataFrame([(f, os.path.getsize(f) / 1e6) for f in sorted(glob.glob(f"{OUT}/**/*", recursive=True)) if os.path.isfile(f)], columns=["file", "MB"])
sz.style.format({"MB": "{:.2f}"})

,file,MB
0,phase4/outputs/error_by_segment.csv,0.00
1,phase4/outputs/error_by_store_family.csv,0.01
2,phase4/outputs/feature_ablation.csv,0.00
3,phase4/outputs/feature_importance_gain.csv,0.01
4,phase4/outputs/feature_importance_shap_F5.csv,0.00
5,phase4/outputs/feature_list.json,0.00
6,phase4/outputs/final_forecast.parquet,0.30
7,phase4/outputs/final_model_config.json,0.01
8,phase4/outputs/fold_scores.csv,0.00
9,phase4/outputs/hyperparameters_tried.json,0.00


## 18. Executive summary

**The question.** Can a machine-learning model beat the planner's simple rule - *"sell what you sold on the same weekday over the last four weeks"* - forecasting daily unit sales 1-16 days ahead for 54 stores x 33 product families? Yes, consistently, across five out-of-time validation windows spanning back-to-school 2016, a holiday-heavy spring, and three recent 2017 windows.

**Technical results (RMSLE, mean of 5 embargoed folds; lower is better).**

| Model | Mean RMSLE | vs baseline (0.4693) |
|---|---|---|
| Baseline - mean of last 4 same weekdays | 0.4693 | - |
| Random Forest (tier 1+2, best of two feature sets) | 0.4058 | +13.5% |
| LightGBM (tier 1+2) | 0.3961 | +15.6% |
| XGBoost (tier 1+2) | 0.3959 | +15.6% |
| **Blend: LightGBM + XGBoost (50/50) - SELECTED** | **0.3942** | **+16.0%** |

The blend beat the baseline on **every one of the 5 folds** (worst fold 0.4139, still far under the baseline's own worst fold of 0.5063), was never more than 0.02 RMSLE behind the single best model on any individual fold, and cost about 2.5 minutes of training per fold on 2 CPU cores - practical for a scheduled retrain. Feature ablation showed tier-1+2 features (78) clearly beat tier-1-only (29); none of the tier-3 experimental features (transactions, oil price, city/state identity, store age, calendar year) produced a gain of at least 0.0005 RMSLE on 4 of 5 folds, so they were **not** adopted - added complexity with no measurable benefit. A "predict the correction to the baseline" (residual-target) variant matched but did not beat the direct target, so the simpler direct formulation was kept.

**Most influential features (gain, LightGBM).** Same-weekday history dominates: `sd_mean_8` (8-week same-weekday mean, 54.1% of gain) and `sd_mean_4` (the baseline's own signal, 26.4%) together explain 80% of what the model uses; `roll_mean_7d` (7.6%) and the most recent same-weekday value `sd_lag_1` (5.5%) round out the top four - together 93.6% of gain. This says the model is mainly a *refinement* of the same-weekday baseline rather than something unrelated to it: it does not discover a completely different signal, it corrects the baseline's blind spots (recent 7-day momentum, the very latest same-weekday reading) - consistent with the modest but real 16% RMSLE gain. Promotion, holiday and calendar features matter but carry small individual shares once the level features are in the model; SHAP confirms the same ranking on an independent 20,000-row sample of the latest fold. None of this should be read as causal - these are the features the model *uses*, not proof that, say, a 7-day rolling mean *causes* next week's sales.

**Where the model is strong.** High-volume, promoted, regular-selling series: PRODUCE, DAIRY and BREAD/BAKERY are the easiest families (RMSLE 0.13-0.15), promoted rows score 0.30 vs 0.45 for non-promoted rows (a 23% gain over baseline vs 13% off-promotion), and the best stores reach RMSLE 0.31. Forecast horizon barely matters: RMSLE is nearly flat from day 1 to day 16 (the baseline's own error grows a little faster), because the model leans on the fixed same-weekday statistics rather than on short-horizon-only signals that decay with distance.

**Where the model is weak.** Low-volume and intermittent series carry the highest per-row RMSLE (0.32-0.63 for series selling ≤5 units/day) even though the *dollar* improvement over baseline there is largest in relative terms; SCHOOL AND OFFICE SUPPLIES, LINGERIE and GROCERY II are the hardest families (RMSLE 0.60-0.66) - low-frequency, spiky, back-to-school- and event-driven demand that a smooth statistical feature set struggles to time exactly. Holiday/event days (1.4% of rows) score worse than regular days (0.439 vs 0.394) and show a small positive bias (the model slightly over-forecasts on holidays) - a caution for holiday-period ordering.

**The dormant-series rule (`rule_zero`).** 4.1% of validation rows (5,690) belong to 98 store-family series with no recent or no history; forecasting them as exactly 0 units is correct 99.7% of the time and improves mean RMSLE only marginally (0.3964 -> 0.3961) because these rows are already near-zero-error under the model - the rule's value is operational (it removes noise from training and guarantees a sane forecast for series a model has never seen sell) more than statistical. In the 0.3% of rows where the rule was wrong, the missed sales were tiny (mean 6 units total per fold, max 3 units in any single row) - not stockout-scale volumes. At the real forecast origin (2017-08-15), 80 of 1,782 series (4.5%, 1,280 of 28,512 rows) are forecast as 0 under this rule; this reflects genuinely dormant or unlisted products, not a claim about inventory or stockouts.

**The final 16-day forecast (2017-08-16 -> 2017-08-31).** The selected blend was refit on all 2,839,036 eligible training rows (origins 2015-07-07 -> 2017-08-08, averaged over 2 random seeds per component) and produced exactly 28,512 predictions - one per store x family x day, clipped at 0, `expm1`-transformed, not rounded. Total forecast volume is 12.56M units, about 4.9% below what the same-weekday baseline would predict for the same 16 days (daily ratio 0.87-1.04) - consistent with the model correcting the baseline downward on average rather than a systematic shift in one direction. GROCERY I, BEVERAGES and PRODUCE account for 68% of forecast volume, matching their share of historical sales.

**Limitations.** (1) Validation folds were also used, in a small way, to pick hyperparameters and the number of boosting rounds (tuned on 3 of the 5 folds), so the reported 16.0% improvement carries a little optimism - true held-out performance on genuinely unseen future data is likely a bit lower. (2) RMSLE is a relative, log-scale error; it does not directly translate to a unit or dollar accuracy target, and a low RMSLE on high-volume families can still miss the exact peak of a promotion by a meaningful number of units. (3) Intermittent and very low-volume series remain hard to forecast in absolute terms, regardless of model choice. (4) The model was trained on 2015-2017 data only; a real deployment would need periodic retraining as promotions, assortment and the economic backdrop (oil-dependent Ecuador) drift. (5) This forecast is a **sales prediction**, not an inventory decision - it does not account for safety stock, lead times, minimum order quantities or supplier constraints, and forecast accuracy alone does not prove that any inventory policy built on top of it would be optimal.

**What this enables for the business.** A demand planner can replace (or blend) the manual same-weekday rule with this forecast to get a systematically better starting point for the next 16 days, with the biggest relative gains exactly where they matter most operationally - promoted items and low-volume series that are hardest to eyeball. The per-store, per-family and per-horizon error breakdown (section 11) also tells a planner *where to still apply manual judgment* - holidays, LINGERIE/SCHOOL AND OFFICE SUPPLIES, and the lowest-volume tail - rather than trusting the model uniformly everywhere. This forecast and its saved artifacts are the input to, but not a substitute for, a downstream inventory-risk decision layer.